In [2]:
# Pull data from the openFDA drug/event endpoint and normalize into a pandas DataFrame
import os
import requests
import pandas as pd

BASE_URL = "https://api.fda.gov/drug/event.json"

# --- Configure your API key ---
# Option 1: set an environment variable FDA_API_KEY
# Option 2: paste your key into the string below
API_KEY = os.getenv("FDA_API_KEY", "")  # optional; openFDA works without a key for most use
# Treat common placeholders as "no key"
if API_KEY.strip().lower() in {"yourapikeyhere", "", "none", "null"}:
    API_KEY = ""

# --- Request parameters ---
# openFDA uses 'limit' (max 100) and 'skip' for pagination
limit = 100
skip = 0
num_pages = 1  # increase (e.g., 5, 10, ...) to pull more pages

all_results = []
for page in range(num_pages):
    params = {
        "limit": limit,
        "skip": skip,
    }
    if API_KEY:
        params["api_key"] = API_KEY
    r = requests.get(BASE_URL, params=params, timeout=30)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        # Provide a more actionable message for common auth/config issues
        raise requests.HTTPError(
            f"Request failed ({r.status_code}). If you intended to use an API key, set env var FDA_API_KEY. URL was: {r.url}"
        ) from e
    payload = r.json()
    results = payload.get("results", [])
    all_results.extend(results)
    skip += limit

print(f"Fetched {len(all_results)} event records")

# Normalize nested JSON to a flat table
df_events = pd.json_normalize(all_results, sep=".")
print(df_events.shape)

# Show a compact preview of commonly useful fields if present
candidate_cols = [
    "safetyreportid",
    "receivedate",
    "receiptdate",
    "serious",
    "seriousnessdeath",
    "patient.patientsex",
    "patient.patientagegroup",
    "patient.reaction",
    "patient.drug",
]
existing = [c for c in candidate_cols if c in df_events.columns]

display(df_events[existing].head(3) if existing else df_events.head(3))

Fetched 100 event records
(100, 39)


,safetyreportid,receivedate,receiptdate,serious,seriousnessdeath,patient.patientsex,patient.patientagegroup,patient.reaction,patient.drug
0,5801206-7,20080707,20080625,1,1,1,NaN,[{'reactionmeddrapt': 'DRUG ADMINISTRATION ERR...,"[{'drugcharacterization': '1', 'medicinalprodu..."
1,10003300,20140306,20140306,1,NaN,2,NaN,"[{'reactionmeddraversionpt': '17.0', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
2,10003301,20140228,20140228,1,NaN,2,NaN,"[{'reactionmeddraversionpt': '17.0', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."


In [3]:
# Load the FAERS_KEY secret and set it as the API key (do not print the secret)
import os
import sphinxai

API_KEY = await sphinxai.get_user_secret_value("FAERS_KEY")
os.environ["FDA_API_KEY"] = API_KEY  # reuse existing code path that reads FDA_API_KEY

# Basic sanity check without revealing the key
assert isinstance(API_KEY, str) and len(API_KEY) > 10

In [4]:
# Confirm whether future openFDA calls will include the API key (without printing it)
import os

api_key_env = os.getenv("FDA_API_KEY", "")
print("FDA_API_KEY is set in env:", bool(api_key_env))
print("Length of FDA_API_KEY:", len(api_key_env) if api_key_env else 0)

# Note: cell 0 captured API_KEY at execution time; it will only use the key if re-run or modified to read env each request.
print("Cell 0 variable API_KEY currently has non-empty value:", bool(globals().get("API_KEY", "")))
print("Cell 0 API_KEY length:", len(globals().get("API_KEY", "")) if globals().get("API_KEY", "") else 0)


FDA_API_KEY is set in env: True
Length of FDA_API_KEY: 40
Cell 0 variable API_KEY currently has non-empty value: True
Cell 0 API_KEY length: 40


In [5]:
# Verify whether the *next* request would include the api_key parameter (without exposing the key)
import os

_api_key = os.getenv("FDA_API_KEY", "").strip()
would_send_key = bool(_api_key) and _api_key.lower() not in {"yourapikeyhere", "none", "null"}

print("Next openFDA request would include api_key param:", would_send_key)
print("Key present in env (non-empty):", bool(_api_key))


Next openFDA request would include api_key param: True
Key present in env (non-empty): True


In [6]:
# Display ~50 random rows from df_events (use a fixed seed for reproducibility)
import pandas as pd

n = 50
seed = 42

# Choose a readable subset of columns (keep nested list columns but don't explode here)
preferred_cols = [
    "safetyreportid",
    "receivedate",
    "receiptdate",
    "serious",
    "seriousnessdeath",
    "seriousnesshospitalization",
    "seriousnesslifethreatening",
    "seriousnessother",
    "occurcountry",
    "patient.patientsex",
    "patient.patientagegroup",
    "patient.patientweight",
    "patient.reaction",
    "patient.drug",
]
cols = [c for c in preferred_cols if c in df_events.columns]

sample_df = df_events.sample(n=min(n, len(df_events)), random_state=seed)[cols].reset_index(drop=True)

# Make the nested columns easier to view by truncating long lists/dicts as strings
nested_cols = [c for c in ["patient.reaction", "patient.drug"] if c in sample_df.columns]
for c in nested_cols:
    sample_df[c] = sample_df[c].astype(str).str.slice(0, 250) + sample_df[c].astype(str).apply(lambda s: "…" if len(s) > 250 else "")

display(sample_df)
print(f"Displayed {len(sample_df)} random rows out of {len(df_events)} total.")


,safetyreportid,receivedate,receiptdate,serious,seriousnessdeath,seriousnesshospitalization,seriousnesslifethreatening,seriousnessother,occurcountry,patient.patientsex,patient.patientagegroup,patient.patientweight,patient.reaction,patient.drug
0,10003384,20140312,20150812,2,NaN,NaN,NaN,NaN,US,1,6,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
1,10003353,20140312,20140312,2,NaN,NaN,NaN,NaN,US,1,NaN,NaN,"[{'reactionmeddraversionpt': '17.0', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
2,10003370,20140312,20150812,2,NaN,NaN,NaN,NaN,US,2,NaN,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
3,10003345,20140312,20140312,1,NaN,1,NaN,NaN,FR,2,NaN,NaN,"[{'reactionmeddraversionpt': '17.0', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
4,10003344,20140312,20150812,2,NaN,NaN,NaN,NaN,US,2,6,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
5,10003339,20140312,20150812,2,NaN,NaN,NaN,NaN,US,1,6,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
6,10003322,20140312,20150812,2,NaN,NaN,NaN,NaN,US,2,5,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
7,10003381,20140312,20150812,2,NaN,NaN,NaN,NaN,US,1,6,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."
8,10003310,20140312,20150812,2,NaN,NaN,NaN,NaN,US,2,6,NaN,"[{'reactionmeddraversionpt': '18.1', 'reaction...","[{'drugcharacterization': '2', 'medicinalprodu..."
9,5801206-7,20080707,20080625,1,1,NaN,NaN,NaN,NaN,1,NaN,NaN,[{'reactionmeddrapt': 'DRUG ADMINISTRATION ERR...,"[{'drugcharacterization': '1', 'medicinalprodu..."


Displayed 50 random rows out of 100 total.


In [7]:
# Show the column names (and count) for df_events
print(f"df_events columns ({len(df_events.columns)}):")
print(list(df_events.columns))

# Optional: quick dtype summary to understand which fields are numeric vs text/nested
display(df_events.dtypes.to_frame("dtype"))

df_events columns (39):
['safetyreportid', 'transmissiondateformat', 'transmissiondate', 'serious', 'seriousnessdeath', 'receivedateformat', 'receivedate', 'receiptdateformat', 'receiptdate', 'fulfillexpeditecriteria', 'companynumb', 'receiver', 'primarysource.reportercountry', 'primarysource.qualification', 'sender.senderorganization', 'patient.patientonsetage', 'patient.patientonsetageunit', 'patient.patientsex', 'patient.patientdeath.patientdeathdateformat', 'patient.patientdeath.patientdeathdate', 'patient.reaction', 'patient.drug', 'safetyreportversion', 'primarysourcecountry', 'reporttype', 'seriousnessdisabling', 'duplicate', 'reportduplicate.duplicatesource', 'reportduplicate.duplicatenumb', 'sender.sendertype', 'receiver.receivertype', 'receiver.receiverorganization', 'seriousnessother', 'occurcountry', 'patient.patientagegroup', 'seriousnesshospitalization', 'patient.summary.narrativeincludeclinical', 'seriousnesslifethreatening', 'patient.patientweight']


,dtype
safetyreportid,object
transmissiondateformat,object
transmissiondate,object
serious,object
seriousnessdeath,object
receivedateformat,object
receivedate,object
receiptdateformat,object
receiptdate,object
fulfillexpeditecriteria,object


## Datasets similar to FAERS **with patient medication history**

FAERS (and EudraVigilance/VigiBase) are *spontaneous reporting systems* and typically **do not contain longitudinal “patient drug history”** beyond the drugs listed in a single report (suspect + concomitant). If you specifically need *drug exposure history over time*, you generally need **EHR or claims** data.

### Best matches (AE/diagnosis + longitudinal medication exposures)

1) **OMOP CDM EHR datasets (OHDSI ecosystem)**
- **What you get:** Person-level longitudinal records: *drug_exposure* (Rx/administration), conditions, procedures, visits, labs (varies). You can define adverse drug reactions as outcomes (e.g., anaphylaxis, SJS/TEN, liver injury) and study drug history prior to the event.
- **Where to get data:**
  - **Synthetic public option:** **SynPUF → OMOP** conversions exist in the community (synthetic claims-like, good for prototyping).
  - **Real-world (restricted):** **MIMIC-IV** can be transformed to OMOP (community ETLs). Also many institutional OMOPs.
- **Why it’s similar:** You can build FAERS-like “cases” but with true exposure history.

2) **MIMIC-IV (critical care EHR, public but credentialed)**
- **What you get:** Detailed ICU EHR data including medications administered, diagnoses, labs, notes.
- **Drug history:** Strong **in-hospital** medication timeline; limited pre-hospital outpatient history.
- **Access:** Requires PhysioNet credentialing + CITI training.
- **Why it’s useful:** Great for medication safety outcomes during hospitalization (e.g., bleeding with anticoagulants, AKI with nephrotoxins).

3) **UK Biobank (restricted, very rich longitudinal health data)**
- **What you get:** Linked EHR/registry data; includes prescriptions in some linked sources; outcomes and longitudinal follow-up.
- **Access:** Application + fees; strong governance.
- **Why it’s useful:** Large cohort for drug-outcome studies with prior history.

4) **Sentinel / PCORnet (distributed networks; generally not fully public)**
- **What you get:** Claims/EHR-based longitudinal drug exposures + outcomes; used by FDA/PCORI for active surveillance.
- **Access:** Typically requires institutional partnership.

### “Similar to FAERS” (spontaneous reporting), but *still limited* drug history

5) **WHO VigiBase** (global ICSRs) and **EMA EudraVigilance** (EU ICSRs)
- **What you get:** Individual Case Safety Reports with suspect and concomitant meds *at the time of report*.
- **Drug history:** Usually not longitudinal; more complete than FAERS in some fields depending on reporter.
- **Access:** VigiBase is licensed; EudraVigilance has public aggregate access and restricted ICSR access for some stakeholders.

### If your “drug history” can be narrative/unstructured

6) **EHR clinical notes datasets (e.g., MIMIC-IV-Note, if available in your environment)**
- **What you get:** Medication history often appears in admission notes, discharge summaries.
- **Approach:** NLP to extract prior meds and link to reaction episodes.

---

## Quick clarification (so I recommend the right thing)
1) Do you need **publicly downloadable** data, or is **credentialed/restricted** access okay (e.g., MIMIC-IV, UK Biobank)?
2) Does “patient drug history” mean **true longitudinal timeline** (meds over months/years), or is “concomitant + prior meds mentioned in the case report” sufficient?

If you answer those two, I can narrow to 1–2 best options and provide concrete download/access steps and (if you want) an example schema mapping for medication history fields.

## Can FAERS be used to infer *pre‑existing conditions*?

**Sometimes, but imperfectly.** FAERS is an ICSR (spontaneous report) system, not a longitudinal EHR. It doesn’t contain a clean, complete “problem list.” However, you can often infer *likely* preexisting conditions from fields that appear in many reports:

### 1) **Drug indication** (best structured proxy)
In many FAERS reports, each drug can include an **indication** (why the drug was taken). Those indications frequently correspond to chronic conditions (e.g., diabetes, hypertension, RA).

- Where it lives (common): `patient.drug[*].drugindication` or sometimes an openFDA-mapped field like `openfda.*` + an indication string.
- Caveats:
  - Not always present.
  - Can reflect the *acute reason* for treatment (e.g., infection) rather than longstanding comorbidity.
  - Free text / non-standard spellings.

### 2) **Medical history / comorbidities fields** (sometimes present, often sparse)
Depending on the version of the case and how it was reported, FAERS can include items akin to:
- `patient.medicalhistory` (or similar),
- `patient.patientmedicalhistoryepisode[*].patientmedicalhistoryepisode`,
- `patient.patientpastdrugtherapy`.

**In openFDA**, these fields may or may not show up depending on the report and what openFDA includes for that record. Even when present, completeness varies a lot.

### 3) **Concomitant medications** (indirect proxy)
If a report lists **concomitant drugs** (not the primary suspect), those meds can imply chronic conditions (e.g., metformin → diabetes).

- Where it lives: `patient.drug[*].drugcharacterization` often distinguishes suspect vs concomitant.
- Caveats:
  - Off-label use and polypharmacy make inference noisy.
  - You usually can’t tell timing/duration—only that the reporter listed the medication.

### 4) **Reaction / diagnosis terms are *not* necessarily preexisting**
`patient.reaction[*].reactionmeddrapt` describes reported adverse events/outcomes and can include conditions, but those are typically **the event**, not baseline comorbidity.

### 5) **Narratives** (if available)
Some reports contain a narrative (free text) that may mention comorbidities.
- In your pull, the closest is: `patient.summary.narrativeincludeclinical` (mostly missing in our 100-row sample).
- Extracting comorbidities from narrative requires NLP and is still incomplete.

---

## Practical guidance
If your goal is “baseline comorbidities,” the **most defensible** FAERS-derived features are:
1) **Indications** for the suspect and concomitant drugs.
2) **Concomitant medication list** (as a proxy for treated conditions).
3) (Optional) **Medical history episodes** when present.

If you tell me whether you want a *quick heuristic* (e.g., top inferred comorbidities) or a *cleaner mapping* (e.g., map indications to MedDRA/SNOMED groups), I can add code to extract those from `df_events` and summarize frequencies.


## Datasets to learn **“pre‑existing history → likely meds”**

What you’re describing is essentially **condition-to-medication imputation / treatment pattern modeling** (e.g., “patients with diabetes + CKD + CHF are likely on metformin/SGLT2/ACEi/loop diuretic…”). To do this credibly you need **patient-level longitudinal prescribing/dispensing** data (EHR or claims). Spontaneous AE systems (FAERS/EudraVigilance) generally won’t work because they don’t contain full medication timelines.

### Best public / semi-public options

1) **CMS DE-SynPUF (Synthetic Medicare claims)** *(public synthetic)*
- **Why it fits:** It’s claims-like longitudinal data with diagnoses + procedure + (depending on file) **prescription/Part D–like drug events** in some distributions/derivatives.
- **Use:** Learn a model mapping diagnosis history → subsequent drug fills.
- **Notes:** Synthetic (not real utilization), but great for prototyping pipelines and feature engineering.

2) **OMOP CDM datasets (OHDSI ecosystem)** *(varies: synthetic/public vs restricted)*
- **Why it fits:** OMOP has a dedicated **`DRUG_EXPOSURE`** table and **`CONDITION_OCCURRENCE`** table. This is the cleanest schema for “history → meds”.
- **Where:**
  - Public/synthetic OMOPs exist (community-maintained).
  - Many real OMOP datasets exist but are restricted (institutional).
- **Use:** Train interpretable models (e.g., multinomial logistic regression) to predict common drug classes given condition vectors.

3) **MIMIC-IV** *(public but credentialed; ICU/inpatient)*
- **Why it fits:** Has longitudinal in-hospital medication administrations/orders + diagnoses.
- **Use:** Good for **inpatient** “conditions during admission → meds administered”.
- **Limitation:** Limited outpatient pre-hospital med history.

### Strong real-world options (restricted, but ideal)

4) **Sentinel / PCORnet / large claims (e.g., MarketScan, Optum)**
- **Why it fits:** Longitudinal diagnoses + pharmacy claims = most direct way to learn prescribing likelihoods.
- **Access:** Generally requires institutional agreements.

5) **UK Biobank (linked EHR + prescriptions where available)**
- **Why it fits:** Large cohort, longitudinal follow-up.
- **Access:** Application + fees.

---

## Quick clarification so I recommend the *best* dataset for your constraints
1) Do you need something **fully public**, or is **credentialed/restricted** access acceptable?
2) Do you want to predict **specific drug ingredients** (e.g., “metformin”) or **drug classes** (e.g., “biguanides”)?

Suggested answers:
- Access: (A) fully public only (B) credentialed OK (C) restricted OK
- Target granularity: (A) ingredient (B) class (C) either


## CMS DE‑SynPUF (Synthetic Medicare Claims) — details

**What it is**
- **DE‑SynPUF** = *Data Entrepreneurs’ Synthetic Public Use File* from **CMS**.
- It is **synthetic (not real patients)** but is designed to resemble Medicare claims structure and code distributions.
- Best for **prototyping** pipelines: feature engineering for diagnosis history, linking to prescription events, cohort building, and model training workflows.

---

## What you can do with it for “history → likely meds”
If you have a SynPUF flavor that includes **drug events (Part D PDE)** (or you use a community OMOP conversion that includes drugs), you can:
1) Build a **patient history window** (e.g., prior 6–24 months) from diagnosis/procedure claims.
2) Define an **index date** (e.g., first occurrence of a condition, a hospitalization, or a fixed calendar split).
3) Predict the **next filled medication** (ingredient, NDC, or drug class) within a prediction horizon.

If your distribution **does not include Part D/PDE**, you can still model *inpatient/outpatient procedure + diagnosis patterns*, but not pharmacy fills.

---

## Typical files / “tables” you’ll see (varies by release)
Most SynPUF distributions are delivered as flat files (CSV) that mimic claims and enrollment:

### 1) Beneficiary / enrollment summary
- One row per synthetic beneficiary per year.
- Contains demographics and eligibility-type variables.
- Used to define **continuous enrollment** and basic covariates (age/sex).

### 2) Carrier (Physician/supplier) claims
- Diagnoses (ICD) and procedures (HCPCS/CPT), service dates, provider info.
- Good for outpatient diagnosis history.

### 3) Outpatient claims
- Facility outpatient encounters, revenue center fields, diagnoses/procedures.

### 4) Inpatient claims
- Hospital admissions/discharges, DRGs, diagnosis/procedure codes.

### 5) (Sometimes) Part D / PDE (Prescription Drug Event)
- This is the key for your task.
- Fields typically include **fill date**, **NDC**, quantity, days supply, etc.

---

## Key features you’d engineer
### Pre‑existing conditions (history)
- **ICD diagnosis codes** from inpatient/outpatient/carrier claims.
- Roll up to:
  - condition groupers (e.g., **CCS**, **Elixhauser**, **Charlson**), or
  - simply a sparse bag-of-codes (top N ICDs).

### Medication targets
- If you have **NDC**: map to
  - ingredient (RxNorm),
  - therapeutic class (e.g., ATC), or
  - custom drug groupings.
- For interpretability, predicting **drug class** is usually easier than raw NDC.

---

## Major limitations (important)
- **Synthetic**: you shouldn’t publish clinical conclusions as if they reflect real-world prescribing.
- **Medicare-like population**: tends to represent older/disabled beneficiaries; prescribing patterns differ from commercial populations.
- **Coding realism varies**: frequency of rare conditions/drugs and multi-morbidity patterns can be imperfect.
- **Not an ADR dataset**: this is claims-like utilization; you *can* define outcomes (e.g., hospitalization for bleeding) but it’s not a spontaneous ADR repository.

---

## Practical “where do I get it?”
Because SynPUF has had multiple distributions over time, access typically comes from:
- CMS historical SynPUF download pages (sometimes moved/archived), **or**
- Community mirrors / tutorials, **or**
- **OMOP-converted SynPUF** datasets shared within the OHDSI community.

If you tell me which route you prefer:
1) **Raw claims files (CSV)**, or
2) **OMOP CDM version**
…I can give you the most direct download + the exact tables/columns to use for the “conditions → drug exposure” task.


## CMS DE-SynPUF (Synthetic Medicare Claims) — deeper dive (what to look for)

### What SynPUF is (and isn’t)
- **Synthetic** Medicare-like claims data published by CMS for **pipeline prototyping**.
- Organized like claims/enrollment extracts: multiple files representing **different claim types**.
- Useful for learning workflows like **“diagnosis history → future drug fills”**, but **not** for drawing real clinical utilization conclusions.

### The *single most important question* for your use-case
To predict “preexisting history → likely meds”, you must have **pharmacy claims**.
- Look for a **Part D / PDE (Prescription Drug Event)** file (sometimes called *PDE*, *PartD*, *Rx*, or *drug events* depending on the distribution).
- If your SynPUF package **does not include PDE**, you can still do “history → procedures/visits”, but not “history → meds filled.”

### Typical file types you may see (varies by release/distribution)
1. **Beneficiary Summary / Enrollment** (1 row per beneficiary per year)
   - Used for demographics, eligibility/coverage, and cohort definitions.
2. **Inpatient** claims
   - Admission/discharge dates + diagnosis/procedure codes.
3. **Outpatient** claims
   - Facility outpatient encounters + codes.
4. **Carrier (Physician/Supplier)** claims
   - Professional claims; good source of outpatient diagnosis history.
5. **Part D / PDE** *(if included)*
   - Drug dispensing events: NDC + service/fill date + quantity/days supply.

### How tables connect (join keys you should expect)
- A **beneficiary ID** (often something like `DESYNPUF_ID` or similar) appears across files.
- Many claim files have a **claim ID** for within-file uniqueness.
- Time alignment is via **service dates** (carrier/outpatient) or **admit/discharge** (inpatient) and **fill date** (PDE).

### What to inspect immediately after obtaining SynPUF
Checklist for the junior DS once files are in hand:
1. Confirm file list includes **PDE/Part D**.
2. For each file: print **row count**, **column list**, **date columns**, **ID columns**.
3. Verify each file has a beneficiary key with **near-100% non-null**.
4. For diagnosis history: count how many diagnosis fields exist (often multiple columns per claim).
5. For PDE: confirm there is an **NDC-like column**, a **fill/service date**, and ideally **days supply**.

### Coding systems / mappings you’ll likely need
- Diagnoses are typically **ICD-9/ICD-10** (depends on year/release).
- Drug codes in PDE are typically **NDC**.
  - For modeling/interpretability, map NDC → **ingredient (RxNorm)** or → **drug class** (ATC / AHFS / custom).
  - This mapping step is often the biggest “plumbing” task.

### Common modeling framing for “history → likely meds”
- Define an **index date** per patient (e.g., end of a lookback window, first diagnosis of interest, or a fixed calendar cut).
- Build **features** from prior claims in a **lookback window** (e.g., 6–12 months of ICD codes or grouped comorbidity indices like Charlson/Elixhauser).
- Define **targets** as the **next** PDE fill(s) within a prediction window (e.g., next 30/90 days), at ingredient or class level.

### Limitations to document explicitly
- Synthetic data: don’t interpret as real prescribing.
- Medicare-like: older/disabled skew.
- Coding density and longitudinal continuity can differ from real claims.

### Clarifying question (decides how you should structure the pipeline)
Do you want to predict:
- **Single next medication** (multiclass), or
- **Top-N likely meds/classes** (multi-label), or
- **Probability by class family** (more interpretable)?


In [10]:
# Document key context about CMS DE-SynPUF and what we still need to verify once files are obtained
from IPython.display import Markdown, display

md = """
## Context gathered so far (CMS DE-SynPUF)

### What SynPUF is
- **CMS DE-SynPUF** is a **synthetic** (not real-patient) Medicare-claims-like dataset intended for **development/prototyping**.
- It is delivered as multiple claim-type files (often CSV) that can be joined by a beneficiary identifier (commonly `DESYNPUF_ID`).

### What we need for the user’s use-case (history → likely meds)
- The critical requirement is the presence of **pharmacy claims** (often a **Part D / PDE** file) with **NDC** and **fill/service date**.
- Without PDE/Part D, the dataset can support history → visits/procedures, but not history → dispensed medications.

### What we can’t verify yet in this notebook
- We do **not** currently have SynPUF files in the workspace, so we can’t confirm:
  - exact file names
  - whether PDE is included
  - exact column names for diagnosis slots, service dates, and NDC

### Assumptions (must be confirmed during file inspection)
- There will be a beneficiary key column like `DESYNPUF_ID` in every file.
- Claim files will have multiple diagnosis columns (e.g., `ICD9_DGNS_CD_1..n`) and date columns.
- PDE file (if present) will have `NDC`-like code, date, quantity, and days supply.

"""

display(Markdown(md))
print("Notebook currently contains only openFDA/FAERS sample data; CMS SynPUF exploration requires obtaining the SynPUF files first.")



## Context gathered so far (CMS DE-SynPUF)

### What SynPUF is
- **CMS DE-SynPUF** is a **synthetic** (not real-patient) Medicare-claims-like dataset intended for **development/prototyping**.
- It is delivered as multiple claim-type files (often CSV) that can be joined by a beneficiary identifier (commonly `DESYNPUF_ID`).

### What we need for the user’s use-case (history → likely meds)
- The critical requirement is the presence of **pharmacy claims** (often a **Part D / PDE** file) with **NDC** and **fill/service date**.
- Without PDE/Part D, the dataset can support history → visits/procedures, but not history → dispensed medications.

### What we can’t verify yet in this notebook
- We do **not** currently have SynPUF files in the workspace, so we can’t confirm:
  - exact file names
  - whether PDE is included
  - exact column names for diagnosis slots, service dates, and NDC

### Assumptions (must be confirmed during file inspection)
- There will be a beneficiary key column like `DESYNPUF_ID` in every file.
- Claim files will have multiple diagnosis columns (e.g., `ICD9_DGNS_CD_1..n`) and date columns.
- PDE file (if present) will have `NDC`-like code, date, quantity, and days supply.



Notebook currently contains only openFDA/FAERS sample data; CMS SynPUF exploration requires obtaining the SynPUF files first.


In [11]:
# Show an example row from the currently loaded openFDA/FAERS sample (df_events)
# and (separately) construct an illustrative mock example of what a CMS DE-SynPUF row often looks like.

import pandas as pd

# --- 1) Real example: one row from df_events (already loaded in this notebook) ---
example_cols = [
    "safetyreportid",
    "receivedate",
    "receiptdate",
    "serious",
    "occurcountry",
    "patient.patientsex",
    "patient.patientagegroup",
    "patient.patientonsetage",
    "patient.patientonsetageunit",
    "patient.patientweight",
    "patient.reaction",
    "patient.drug",
]
example_cols = [c for c in example_cols if c in df_events.columns]

real_example = df_events.sample(1, random_state=0)[example_cols].copy()
# Make nested columns readable
for c in [c for c in ["patient.reaction", "patient.drug"] if c in real_example.columns]:
    real_example[c] = real_example[c].astype(str).str.slice(0, 400) + real_example[c].astype(str).apply(
        lambda s: "…" if len(s) > 400 else ""
    )

print("Real example row from openFDA FAERS (df_events):")
display(real_example)

# --- 2) Illustrative example: what CMS DE-SynPUF rows commonly look like ---
# NOTE: This is NOT pulled from CMS (we don't have SynPUF files in the workspace).
# It is a representative mock row with typical column patterns.

mock_beneficiary_row = {
    "DESYNPUF_ID": "0000F0D0E0C0B0A0",
    "BENE_BIRTH_DT": "1933-07-14",
    "BENE_SEX_IDENT_CD": "2",
    "BENE_RACE_CD": "1",
    "STATE_CODE": "05",
    "SP_STATE_CODE": "05",
    "BENE_DEATH_DT": "",
    "BENE_HI_CVRAGE_TOT_MONS": 12,
    "BENE_SMI_CVRAGE_TOT_MONS": 12,
}

mock_pde_row = {
    "DESYNPUF_ID": "0000F0D0E0C0B0A0",
    "PDE_ID": "PDE1234567890",
    "SRVC_DT": "2009-03-22",
    "PROD_SRVC_ID": "00093015001",  # NDC-like
    "DAYS_SUPLY_NUM": 30,
    "QTY_DSPNSD_NUM": 60,
}

mock_claim_row = {
    "DESYNPUF_ID": "0000F0D0E0C0B0A0",
    "CLM_ID": "CLM987654321",
    "CLM_FROM_DT": "2009-02-10",
    "CLM_THRU_DT": "2009-02-12",
    "ICD9_DGNS_CD_1": "25000",  # diabetes mellitus without mention of complication
    "ICD9_DGNS_CD_2": "4019",   # essential hypertension
    "ICD9_PRCDR_CD_1": "8154",
}

print("\nIllustrative (mock) examples of common CMS DE-SynPUF row shapes:")
display(pd.DataFrame([mock_beneficiary_row]))
display(pd.DataFrame([mock_claim_row]))
display(pd.DataFrame([mock_pde_row]))

print(
    "\nIf you upload / point me to your SynPUF files (or list the filenames in your folder), "
    "I can pull an actual row from the real CMS dataset in the same way we sampled df_events."
)


Real example row from openFDA FAERS (df_events):


,safetyreportid,receivedate,receiptdate,serious,occurcountry,patient.patientsex,patient.patientagegroup,patient.patientonsetage,patient.patientonsetageunit,patient.patientweight,patient.reaction,patient.drug
26,10003326,20140312,20140312,1,CN,2,NaN,29,801,NaN,"[{'reactionmeddraversionpt': '17.0', 'reaction...","[{'drugcharacterization': '1', 'medicinalprodu..."



Illustrative (mock) examples of common CMS DE-SynPUF row shapes:


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,STATE_CODE,SP_STATE_CODE,BENE_DEATH_DT,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS
0,0000F0D0E0C0B0A0,1933-07-14,2,1,05,05,,12,12


,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_PRCDR_CD_1
0,0000F0D0E0C0B0A0,CLM987654321,2009-02-10,2009-02-12,25000,4019,8154


,DESYNPUF_ID,PDE_ID,SRVC_DT,PROD_SRVC_ID,DAYS_SUPLY_NUM,QTY_DSPNSD_NUM
0,0000F0D0E0C0B0A0,PDE1234567890,2009-03-22,00093015001,30,60



If you upload / point me to your SynPUF files (or list the filenames in your folder), I can pull an actual row from the real CMS dataset in the same way we sampled df_events.


## Example row from CMS DE-SynPUF (what’s possible in this notebook)

- **Right now, this notebook does not have any CMS DE‑SynPUF files loaded**, so we **cannot** sample a *real* SynPUF row yet.
- What we *can* do today:
  1) Show a **real** row from the dataset we *do* have loaded (**openFDA/FAERS**: `df_events`).
  2) Show an **illustrative mock** SynPUF row (typical columns) so you can see the *shape*.

If you want a **real SynPUF** row next:
- Upload the SynPUF files into the workspace (or tell me the folder path + filenames).
- Then we’ll run a tiny snippet to read (e.g.) the **Beneficiary Summary** file and/or the **PDE (Part D)** file and `sample(1)`.

Example snippet to run once the files exist (we’ll tailor `FILEPATH` + delimiter based on your files):
- `df = pd.read_csv(FILEPATH)`
- `display(df.sample(1, random_state=0))`
